# Phase 7 - Evaluation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import time
import uuid
import sqlite3
import hashlib
import statistics
import traceback

import numpy as np
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

EVAL.mkdir(parents=True, exist_ok=True)

GOLD_DIR = EVAL / "gold"
RESULT_DIR = EVAL / "results"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Evaluation:", EVAL)
print("Golden sets:", GOLD_DIR)
print("Results:", RESULT_DIR)

assert ROOT.exists()
assert (PROC / "cfpb_holdout_june_2026.csv").exists()
assert (INDEX / "reg_vectors.npy").exists()
assert (INDEX / "case_vectors.npy").exists()

print("PASS — Evaluation directories and source artifacts verified.")

Project: /home/sagemaker-user/complaintiq
Evaluation: /home/sagemaker-user/complaintiq/eval
Golden sets: /home/sagemaker-user/complaintiq/eval/gold
Results: /home/sagemaker-user/complaintiq/eval/results
PASS — Evaluation directories and source artifacts verified.


In [23]:
from pathlib import Path
import json
import ast
import re

ROOT = Path("/home/sagemaker-user/complaintiq")
NOTEBOOK_DIR = ROOT / "notebooks"

SOURCE_NOTEBOOKS = {
    "Notebook3": NOTEBOOK_DIR / "03_retrieval.ipynb",
    "Notebook4": NOTEBOOK_DIR / "04_agents.ipynb",
    "Notebook5": NOTEBOOK_DIR / "05_graph.ipynb",
}

NOTEBOOK_CATALOG = {}

for notebook_name, notebook_path in SOURCE_NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Notebook not found: {notebook_path}"
    )

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    catalog = []

    for cell_number, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        # Skip notebook magic commands for catalog parsing
        clean_source = "\n".join(
            line for line in source.splitlines()
            if not line.lstrip().startswith(("%", "!"))
        )

        try:
            tree = ast.parse(clean_source)
        except SyntaxError:
            continue

        definitions = []

        for node in tree.body:

            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                definitions.append(f"function: {node.name}")

            elif isinstance(node, ast.ClassDef):
                definitions.append(f"class: {node.name}")

            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                definitions.append("import")

            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = (
                    node.targets if isinstance(node, ast.Assign)
                    else [node.target]
                )

                for target in targets:
                    if isinstance(target, ast.Name):
                        definitions.append(f"variable: {target.id}")

        if definitions:
            catalog.append({
                "cell_number": cell_number,
                "definitions": definitions,
                "source": source
            })

    NOTEBOOK_CATALOG[notebook_name] = catalog

    print(f"\n{'=' * 65}")
    print(notebook_name)
    print(f"{'=' * 65}")

    for item in catalog:
        print(
            f"Cell {item['cell_number']:>3}: "
            + ", ".join(item["definitions"])
        )

print("\nPASS — Saved notebook definitions inventoried.")


Notebook3
Cell   1: import, variable: PROJECT_ROOT
Cell   2: import, import, variable: PROJECT_ROOT, import
Cell   3: import, import, import, import, variable: PROJECT, variable: DATA, variable: PROC, variable: INDEX, variable: REPORTS, variable: EVAL, variable: REGION, variable: BUCKET, variable: S3_INDEX
Cell   5: import, import, variable: PROJECT, variable: PROC, variable: INDEX_DIR, variable: S3_INDEX, variable: required, variable: doc_count, variable: pool_df, variable: holdout_df, variable: pool_count, variable: holdout_count, variable: pool_dates, variable: holdout_dates
Cell   7: import, import
Cell   8: import, variable: AWS_REGION, variable: session, variable: brt
Cell   9: variable: test_body, variable: response, variable: test_vector
Cell  11: variable: CACHE, function: cache_path, function: embed_one, function: embed
Cell  12: function: read_jsonl, variable: reg_chunks, variable: chunk_ids
Cell  14: variable: chunk_lengths
Cell  15: import, import, import, import, import,

In [3]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

# Verify S3 access
s3.head_bucket(Bucket=BUCKET)

# Verify Bedrock client configuration
assert brt.meta.region_name == REGION

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Models:", MODELS)
print("PASS — AWS clients restored.")

Region: us-east-1
Bucket: sagemaker-complaintiq-456427397380
Models: {'worker': 'us.anthropic.claude-sonnet-4-6', 'judge': 'us.anthropic.claude-opus-4-8', 'fast': 'us.anthropic.claude-sonnet-4-6', 'embed': 'amazon.titan-embed-text-v2:0'}
PASS — AWS clients restored.


In [25]:
# ============================================================
# COMPLAINTIQ PHASE 7 — NOTEBOOK 6 BOOTSTRAP
# Restore validated implementations from Notebooks 3, 4 and 5
# No corpus embedding generation, benchmark calls or publishing
# ============================================================

import sys
import os
import json
import ast
import uuid
import re
import hashlib
import sqlite3
import importlib
import inspect

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import boto3
import faiss
import duckdb
import sqlglot

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict
import operator

from botocore.exceptions import ClientError
from langgraph.types import interrupt


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/home/sagemaker-user/complaintiq")

DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

for path in [DATA, PROC, INDEX, STATE, SRC, EVAL]:
    path.mkdir(parents=True, exist_ok=True)

REGION = "us-east-1"
AWS_REGION = REGION
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

sys.path.insert(0, str(SRC))

print("PASS — Project and AWS configuration initialized.")


# ------------------------------------------------------------
# 2. RESTORE PHASE 4 LLM MODULE
# ------------------------------------------------------------

import llm
importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.chat_json)
assert callable(llm.get_usage_report)

print("PASS — Phase 4 LLM wrapper restored.")


# ------------------------------------------------------------
# 3. RESTORE PHASE 3 RETRIEVAL FROM NOTEBOOK 5 CELL 19
# ------------------------------------------------------------

NB5_PATH = ROOT / "notebooks" / "05_graph.ipynb"

assert NB5_PATH.exists(), f"Missing notebook: {NB5_PATH}"

with open(NB5_PATH, "r", encoding="utf-8") as f:
    nb5 = json.load(f)

retrieval_source = "".join(
    nb5["cells"][19]["source"]
)

# Execute the validated original retrieval restoration cell.
# It loads saved vectors and reconstructs the indexes.
# It does not regenerate corpus embeddings.

exec(
    compile(
        retrieval_source,
        "notebook5_cell19_retrieval_restore",
        "exec"
    ),
    globals()
)

assert callable(embed_one)
assert callable(embed)
assert callable(tok)
assert callable(HybridIndex)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval layer restored.")


# ------------------------------------------------------------
# 4. RESTORE PHASE 5 AGENTS FROM NOTEBOOK 5 CELL 14
# ------------------------------------------------------------

agent_restore_source = "".join(
    nb5["cells"][14]["source"]
)

# This is the existing selective AST-based restoration
# mechanism from Notebook 5.

exec(
    compile(
        agent_restore_source,
        "notebook5_cell14_agent_restore",
        "exec"
    ),
    globals()
)

print("PASS — Phase 5 agent restoration executed.")


# ------------------------------------------------------------
# 5. RESTORE NOTEBOOK 5 ROUTING HELPERS
# ------------------------------------------------------------

routing_source = "".join(
    nb5["cells"][22]["source"]
)

exec(
    compile(
        routing_source,
        "notebook5_cell22_routing",
        "exec"
    ),
    globals()
)

assert callable(after_analyst)
assert callable(after_critic)
assert callable(after_human)

print("PASS — Graph routing helpers restored.")


# ------------------------------------------------------------
# 6. RESTORE PUBLISHING HELPERS
# ------------------------------------------------------------

# Restore S3 publishing configuration.
# This cell only defines configuration and does not publish.

publishing_config_source = "".join(
    nb5["cells"][25]["source"]
)

exec(
    compile(
        publishing_config_source,
        "notebook5_cell25_publish_config",
        "exec"
    ),
    globals()
)

# Restore usage, audit and publishing functions.
# Definitions are loaded; none are invoked.

publishing_helpers_source = "".join(
    nb5["cells"][26]["source"]
)

exec(
    compile(
        publishing_helpers_source,
        "notebook5_cell26_publish_helpers",
        "exec"
    ),
    globals()
)

publishing_node_source = "".join(
    nb5["cells"][30]["source"]
)

exec(
    compile(
        publishing_node_source,
        "notebook5_cell30_publish_node",
        "exec"
    ),
    globals()
)

assert callable(get_usage_snapshot)
assert callable(append_audit_record)
assert callable(publish_node)

print("PASS — Publishing helper definitions restored.")
print("NOTE — No S3 report or audit write was performed.")


# ------------------------------------------------------------
# 7. VALIDATE REQUIRED AGENT FUNCTIONS
# ------------------------------------------------------------

REQUIRED_FUNCTIONS = [
    "guard_node",
    "router_node",
    "triage_node",
    "analyst_node",
    "cases_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
    "after_analyst",
    "after_critic",
    "after_human",
    "publish_node",
]

for name in REQUIRED_FUNCTIONS:
    obj = globals().get(name)

    if not callable(obj):
        raise RuntimeError(
            f"Bootstrap validation failed: {name} is missing."
        )

    print(f"PASS | {name}")


# ------------------------------------------------------------
# 8. VALIDATE SHARED STATE AND RETRIEVAL
# ------------------------------------------------------------

assert "CIQState" in globals()
assert "TriageResult" in globals()

assert len(TAXONOMY) > 0

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)


# ------------------------------------------------------------
# 9. FINAL BOOTSTRAP REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPLAINTIQ PHASE 7 BOOTSTRAP VALIDATION")
print("=" * 65)

print("LLM wrapper:          PASS")
print("Embedding functions:  PASS")
print("HybridIndex:           PASS")
print("Regulatory index:     ", REG_INDEX.faiss.ntotal)
print("Complaint index:      ", CASE_INDEX.faiss.ntotal)
print("CIQState:              PASS")
print("Triage taxonomy:      ", len(TAXONOMY))
print("Agent functions:      ", len(REQUIRED_FUNCTIONS))
print("Routing helpers:       PASS")
print("Publishing helpers:    PASS")

print("\nPASS — PHASE 7 RUNTIME RESTORED.")

PASS — Project and AWS configuration initialized.
PASS — Phase 4 LLM wrapper restored.
PASS — All four existing retrieval artifacts found.
Regulatory metadata: 1784
Regulatory vectors: (1784, 1024)
Complaint metadata: 3000
Complaint vectors: (3000, 1024)

PASS — Regulatory index: 1784
PASS — Complaint index: 3000

PASS — Phase 3 retrieval layer restored.
PASS — Phase 3 retrieval layer restored.
Executed implementation cells: [44, 46, 48, 50, 52, 60, 61, 62, 63, 67]
PASS | CIQState
PASS | guard_node
PASS | router_node
PASS | triage_node
PASS | analyst_node
PASS | cases_node
PASS | research_node
PASS | drafter_node
PASS | critic_node
PASS | human_node
PASS | refuse_node
PASS — Phase 5 agent restoration executed.
PASS — All graph-routing functions validated.
PASS — Graph routing helpers restored.
BUCKET: sagemaker-complaintiq-456427397380
REPORT PREFIX: complaintiq/reports
AUDIT KEY: complaintiq/reports/audit.jsonl
PASS — Publishing configuration restored.
PASS — Usage snapshot and audit 

In [26]:
print("CIQState annotations:")
print(CIQState.__annotations__)

CIQState annotations:
{'request': <class 'str'>, 'clean_request': <class 'str'>, 'flags': list[str], 'intent': typing.Literal['TRIAGE', 'ANALYTICS', 'REG_QA', 'BRIEF', 'OUT_OF_SCOPE'], 'triage': <class 'dict'>, 'similar_cases': list[dict], 'sql': <class 'str'>, 'sql_rows': list[dict], 'evidence': list[dict], 'draft': <class 'str'>, 'critique': <class 'dict'>, 'revisions': <class 'int'>, 'human_decision': <class 'dict'>, 'report_uri': <class 'str'>, 'trace': typing.Annotated[list[dict], <built-in function add>]}


In [27]:
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

In [28]:
print("=== FINAL DEPENDENCY CHECK ===")

checks = [
    ("embed_one", callable(embed_one)),
    ("embed", callable(embed)),
    ("tok", callable(tok)),
    ("HybridIndex", callable(HybridIndex)),
    ("REG_INDEX", REG_INDEX.faiss.ntotal == 1784),
    ("CASE_INDEX", CASE_INDEX.faiss.ntotal == 3000),
    ("CIQState", "CIQState" in globals()),
    ("TriageResult", "TriageResult" in globals()),
    ("llm.chat", callable(llm.chat)),
    ("triage_node", callable(triage_node)),
    ("analyst_node", callable(analyst_node)),
    ("research_node", callable(research_node)),
    ("critic_node", callable(critic_node)),
    ("publish_node", callable(publish_node)),
]

for name, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(passed for _, passed in checks)

print("\nALL DEPENDENCY CHECKS PASSED.")

=== FINAL DEPENDENCY CHECK ===
PASS | embed_one
PASS | embed
PASS | tok
PASS | HybridIndex
PASS | REG_INDEX
PASS | CASE_INDEX
PASS | CIQState
PASS | TriageResult
PASS | llm.chat
PASS | triage_node
PASS | analyst_node
PASS | research_node
PASS | critic_node
PASS | publish_node

ALL DEPENDENCY CHECKS PASSED.


In [5]:

import duckdb

# --------------------------------------------------
# Phase 7 — Evaluation database validation
# --------------------------------------------------

DB_PATH = ROOT / "data" / "complaintiq.duckdb"

assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

print("DuckDB:", DB_PATH)

# Close any existing read-only connection from this notebook
if "ro" in globals():
    try:
        ro.close()
    except Exception:
        pass

# Open database in read-only mode
ro = duckdb.connect(str(DB_PATH), read_only=True)

# Verify complaint view
view_check = ro.execute("""
    SELECT COUNT(*)
    FROM v_complaints
""").fetchone()[0]

print("v_complaints rows:", view_check)

assert view_check > 0, "v_complaints is empty"

print("PASS — Read-only evaluation database restored.")


DuckDB: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb
v_complaints rows: 37923
PASS — Read-only evaluation database restored.


In [6]:
sys.path.insert(0, str(SRC))

import importlib
import llm

importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.get_usage_report)

print(json.dumps(
    llm.get_usage_report(),
    indent=2,
    default=str
))

print("PASS — Phase 4 LLM wrapper restored.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}
PASS — Phase 4 LLM wrapper restored.


In [9]:
class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]



In [11]:
assert "HybridIndex" in globals()
assert callable(HybridIndex)

print("PASS — HybridIndex class restored.")

PASS — HybridIndex class restored.


In [13]:
import re

def tok(text):
    """
    Tokenize text for BM25 retrieval.
    Uses the same lowercase alphanumeric tokenization
    expected by the retrieval index.
    """
    return re.findall(r"\b\w+\b", str(text).lower())

print("PASS — tok() function defined.")
print("Test:", tok("12 CFR 1005.11 Electronic Fund Transfers"))

PASS — tok() function defined.
Test: ['12', 'cfr', '1005', '11', 'electronic', 'fund', 'transfers']


In [14]:
assert callable(tok)
assert callable(HybridIndex)

print("PASS — HybridIndex dependencies available.")

PASS — HybridIndex dependencies available.


In [15]:
import re
import faiss
from collections import defaultdict
from rank_bm25 import BM25Okapi

reg_chunks = [
    json.loads(line)
    for line in (INDEX / "reg_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

case_chunks = [
    json.loads(line)
    for line in (INDEX / "case_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

reg_vectors = np.load(INDEX / "reg_vectors.npy")
case_vectors = np.load(INDEX / "case_vectors.npy")

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)

# Re-execute the validated HybridIndex class and embed() implementation
# from Notebook 5's saved source cells before running this cell.

REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval indexes reconstructed.")

PASS — Phase 3 retrieval indexes reconstructed.


In [17]:
def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


In [20]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import numpy as np

print("PASS — Embedding dependencies imported.")

PASS — Embedding dependencies imported.


In [29]:
print("=== REGULATORY RETRIEVAL TEST ===")

results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

for i, item in enumerate(results, 1):
    print(
        i,
        item.get("chunk_id"),
        str(item.get("text", ""))[:180]
    )

assert len(results) > 0

print("\nPASS — Regulatory retrieval operational.")

=== REGULATORY RETRIEVAL TEST ===


Embedding: 100%|██████████| 1/1 [00:00<00:00,  3.99it/s]


1 exam_manual_v2_2012_page_0700_chunk_0001 CFPB Consumer Laws and Regulations EFTA An error includes any of the following: • An unauthorized EFT. • An incorrect EFT to or from the consumer’s account. • The omission from a p
2 exam_manual_v2_2012_page_0734_chunk_0001 CFPB Examination Checklist EFTA CFPB Manual V.2 (October 2012) Checklist 9 Yes No NA 35. If the financial institution requires written confirmation of an error within 10 business d
3 exam_manual_v2_2012_page_0699_chunk_0002 an EFT, for example, when the card is used to draw cash advances directly from a credit line, only the error resolution provisions of Regulation Z will apply. • If the unauthorized
4 12CFR1026_1026.13_chunk_0004 ) Promptly reports that the amount or account is in dispute; (ii) Mails or delivers to the consumer (at the same time the report is made) a written notice of the name and address o
5 exam_manual_v2_2012_page_0735_chunk_0001 Yes No NA CFPB Examination Checklist EFTA 39. If a bill ing error occurre

In [30]:
# ============================================================
# PHASE 7 — G1: JUNE HOLDOUT DATASET INSPECTION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# Paths
GOLD_DIR = EVAL / "gold"
RESULTS_DIR = EVAL / "results"
ANALYSIS_DIR = EVAL / "analysis"
MANIFEST_DIR = EVAL / "manifests"

for directory in [
    GOLD_DIR,
    RESULTS_DIR,
    ANALYSIS_DIR,
    MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

HOLDOUT_FILE = PROC / "cfpb_holdout_june_2026.csv"

assert HOLDOUT_FILE.exists(), (
    f"June holdout file not found: {HOLDOUT_FILE}"
)

# Load holdout dataset
holdout_df = pd.read_csv(HOLDOUT_FILE)

print("=" * 65)
print("G1 — JUNE HOLDOUT DATASET INSPECTION")
print("=" * 65)

print("\nDataset shape:")
print(holdout_df.shape)

print("\nAvailable columns:")
for column in holdout_df.columns:
    print(f"- {column}")

print("\nData types:")
print(holdout_df.dtypes)

print("\nFirst five records:")
display(holdout_df.head())

print("\nMissing values:")
display(
    holdout_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDuplicate records:")
print(holdout_df.duplicated().sum())

print("\nPASS — June holdout dataset loaded.")

G1 — JUNE HOLDOUT DATASET INSPECTION

Dataset shape:
(6110, 16)

Available columns:
- Date received
- Product
- Sub-product
- Issue
- Sub-issue
- Consumer complaint narrative
- Company public response
- Company
- State
- ZIP code
- Tags
- Submitted via
- Date sent to company
- Company response to consumer
- Timely response?
- Complaint ID

Data types:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

First f

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-06-17,Credit card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms,I am filing this complaint because Barclays ha...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MN,554XX,NaN,Web,2026-07-30,Closed with monetary relief,Yes,23300227
1,2026-06-23,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,Despite five separate contacts with Citi betwe...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MO,65109,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23472102
2,2026-06-25,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,"My name is XXXX XXXX, I am a resident of the c...",Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",FL,32824,NaN,Web,2026-06-25,Closed with explanation,Yes,23576641
3,2026-06-02,Credit card,General-purpose credit card or charge card,Getting a credit card,Problem getting a working replacement card,These r proofs they r keeping my money they wo...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,AZ,85051,NaN,Web,2026-06-02,Closed with explanation,Yes,22815215
4,2026-06-23,Checking or savings account,Checking account,Managing an account,Problem accessing account,Since my last complaints the company has start...,Company has responded to the consumer and the ...,BMO BANK NATIONAL ASSOCIATION,DE,198XX,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23466116



Missing values:


,missing_count
Tags,4886
Company public response,3420
State,49
ZIP code,42
Sub-issue,7
Issue,0
Date received,0
Product,0
Company,0
Consumer complaint narrative,0



Duplicate records:
0

PASS — June holdout dataset loaded.


In [31]:
required = [
    "ROOT", "BUCKET", "MODELS", "s3", "brt",
    "con", "ro", "llm",
    "REG_INDEX", "CASE_INDEX",
    "CIQState",
    "guard_node", "router_node",
    "triage_node", "analyst_node",
    "cases_node", "research_node",
    "drafter_node", "critic_node",
    "human_node", "refuse_node"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing evaluation dependencies: {missing}")

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

print("PASS — Notebook 6 runtime is ready.")

PASS — Notebook 6 runtime is ready.


In [33]:
# ============================================================
# PHASE 7 — G1 CELL 9
# REPRODUCIBLE, PRODUCT-STRATIFIED SAMPLING
# ============================================================

import numpy as np
import pandas as pd

G1_SAMPLE_SIZE = 60
G1_RANDOM_SEED = 20261106

required_columns = [
    "Complaint ID",
    "Product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative"
]

missing_columns = [
    c for c in required_columns
    if c not in holdout_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Only use records with a usable narrative.
g1_source = holdout_df.copy()

g1_source["Consumer complaint narrative"] = (
    g1_source["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
    .str.strip()
)

g1_source = g1_source[
    g1_source["Consumer complaint narrative"].str.len() > 0
].copy()

# Complaint IDs must be unique.
assert g1_source["Complaint ID"].is_unique, (
    "Duplicate Complaint IDs detected in source data."
)

assert len(g1_source) >= G1_SAMPLE_SIZE

# Product-level proportional allocation using largest remainder.
product_counts = (
    g1_source.groupby("Product")
    .size()
    .sort_index()
)

raw_allocations = (
    product_counts / product_counts.sum()
) * G1_SAMPLE_SIZE

allocations = np.floor(raw_allocations).astype(int)

remaining = G1_SAMPLE_SIZE - allocations.sum()

remainders = (
    raw_allocations - allocations
).sort_values(ascending=False)

for product in remainders.index:
    if remaining == 0:
        break

    if allocations[product] < product_counts[product]:
        allocations[product] += 1
        remaining -= 1

assert allocations.sum() == G1_SAMPLE_SIZE

# Sample independently within each product.
sampled_groups = []

for product, sample_count in allocations.items():
    if sample_count == 0:
        continue

    group = g1_source[
        g1_source["Product"] == product
    ]

    sampled_groups.append(
        group.sample(
            n=int(sample_count),
            random_state=G1_RANDOM_SEED
        )
    )

g1_sample = (
    pd.concat(sampled_groups)
    .sample(
        frac=1,
        random_state=G1_RANDOM_SEED
    )
    .reset_index(drop=True)
)

assert len(g1_sample) == G1_SAMPLE_SIZE
assert g1_sample["Complaint ID"].is_unique

print("=" * 65)
print("G1 — STRATIFIED SAMPLE")
print("=" * 65)

print("Sample size:", len(g1_sample))
print("Random seed:", G1_RANDOM_SEED)

print("\nProduct allocation:")
display(
    pd.DataFrame({
        "source_count": product_counts,
        "sample_count": allocations
    }).fillna(0).astype(int)
)

print("\nSample distribution:")
display(
    g1_sample["Product"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nIssue distribution:")
display(
    g1_sample["Issue"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nPASS — 60 reproducible G1 records selected.")

G1 — STRATIFIED SAMPLE
Sample size: 60
Random seed: 20261106

Product allocation:


,source_count,sample_count
Product,,
Checking or savings account,3270,32
Credit card,2840,28



Sample distribution:


,sample_count
Product,
Checking or savings account,32
Credit card,28



Issue distribution:


,sample_count
Issue,
Managing an account,18
Fees or interest,6
Problem with a purchase shown on your statement,6
Problem with a lender or other company charging your account,5
Closing an account,5
"Other features, terms, or problems",4
Closing your account,3
Improper use of your report,3
Opening an account,2



PASS — 60 reproducible G1 records selected.


In [34]:
# ============================================================
# PHASE 7 — G1 CELL 10
# BUILD GOLD EVALUATION RECORDS
# ============================================================

import json
import pandas as pd

def clean_gold_value(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    return value


g1_gold_records = []

for _, row in g1_sample.iterrows():

    record = {
        "case_id": str(int(row["Complaint ID"])),

        "request": clean_gold_value(
            row["Consumer complaint narrative"]
        ),

        "expected": {
            "product": clean_gold_value(row["Product"]),
            "issue": clean_gold_value(row["Issue"]),
            "sub_issue": clean_gold_value(row["Sub-issue"])
        },

        # These fields are not supplied as ground truth
        # by the CFPB dataset. They require human annotation.
        "manual_annotation": {
            "severity": None,
            "vulnerable_customer": None,
            "vulnerability_signals": None,
            "regulation_hints": None,
            "summary": None
        },

        "source": {
            "dataset": "cfpb_holdout_june_2026",
            "complaint_id": str(int(row["Complaint ID"])),
            "date_received": clean_gold_value(
                row["Date received"]
            )
        },

        "annotation_status": "pending"
    }

    g1_gold_records.append(record)

assert len(g1_gold_records) == 60

print("=" * 65)
print("G1 — GOLD RECORD GENERATION")
print("=" * 65)

print("Records generated:", len(g1_gold_records))

print("\nExample gold record:")
print(
    json.dumps(
        g1_gold_records[0],
        indent=2,
        ensure_ascii=False
    )
)

print("\nPASS — G1 gold records generated.")

G1 — GOLD RECORD GENERATION
Records generated: 60

Example gold record:
{
  "case_id": "23266147",
  "request": "I am filing this complaint against BMO Bank regarding an unreasonable hold placed on my deposited insurance settlement funds and the overdraft fees that resulted from the banks actions. On XX/XX/year>, I deposited a settlement check from XXXX XXXX at the BMO branch located at XXXX XXXX XXXX, XXXX XXXX, Missouri. BMO placed a hold on the funds but failed to provide adequate notice explaining the reason for the hold, the length of the hold, or my rights under the banks funds availability policy. After the deposit, documentation was provided to verify the legitimacy of the settlement funds. The branch also received verification directly from XXXX XXXX confirming that the check was valid and legitimate. Despite receiving this verification, BMO continued to restrict access to my funds. Because BMO withheld access to verified funds, transactions that would have otherwise been cove

In [35]:
# ============================================================
# PHASE 7 — G1 CELL 11
# HUMAN ANNOTATION TEMPLATE
# ============================================================

import pandas as pd

annotation_rows = []

for record in g1_gold_records:

    annotation_rows.append({
        "case_id": record["case_id"],

        "severity": "",
        "vulnerable_customer": "",
        "vulnerability_signals": "",
        "regulation_hints": "",
        "summary": "",

        "reviewer": "",
        "review_notes": "",
        "annotation_status": "PENDING"
    })

g1_annotation_df = pd.DataFrame(annotation_rows)

G1_ANNOTATION_FILE = (
    GOLD_DIR / "G1_triage_manual_annotations.csv"
)

print("=" * 65)
print("G1 — HUMAN ANNOTATION TEMPLATE")
print("=" * 65)

print("Annotation records:", len(g1_annotation_df))

display(g1_annotation_df.head(10))

print("\nSeverity values to use:")
print(["LOW", "MEDIUM", "HIGH", "CRITICAL"])

print("\nVulnerable customer values to use:")
print(["True", "False"])

print("\nAnnotation file target:")
print(G1_ANNOTATION_FILE)

print("\nPASS — Annotation template prepared.")

G1 — HUMAN ANNOTATION TEMPLATE
Annotation records: 60


,case_id,severity,vulnerable_customer,vulnerability_signals,regulation_hints,summary,reviewer,review_notes,annotation_status
0,23266147,,,,,,,,PENDING
1,23172941,,,,,,,,PENDING
2,23492548,,,,,,,,PENDING
3,23304250,,,,,,,,PENDING
4,22965626,,,,,,,,PENDING
5,23082821,,,,,,,,PENDING
6,23494747,,,,,,,,PENDING
7,23053378,,,,,,,,PENDING
8,22988407,,,,,,,,PENDING
9,22794522,,,,,,,,PENDING



Severity values to use:
['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

Vulnerable customer values to use:
['True', 'False']

Annotation file target:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv

PASS — Annotation template prepared.


In [36]:
# ============================================================
# PHASE 7 — G1 CELL 12
# SAVE GOLD DATASET
# ============================================================

import json
import hashlib
from pathlib import Path

G1_GOLD_FILE = GOLD_DIR / "G1_triage_gold.jsonl"

# Write JSONL.
with open(G1_GOLD_FILE, "w", encoding="utf-8") as f:

    for record in g1_gold_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

# Save human annotation template.
g1_annotation_df.to_csv(
    G1_ANNOTATION_FILE,
    index=False
)

# Calculate SHA256 checksums.
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


print("=" * 65)
print("G1 — SAVED ARTIFACTS")
print("=" * 65)

print("Gold dataset:")
print(G1_GOLD_FILE)
print("Size:", G1_GOLD_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_GOLD_FILE))

print("\nAnnotation template:")
print(G1_ANNOTATION_FILE)
print("Size:", G1_ANNOTATION_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 artifacts saved locally.")

G1 — SAVED ARTIFACTS
Gold dataset:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_gold.jsonl
Size: 116918 bytes
SHA256: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984

Annotation template:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv
Size: 1564 bytes
SHA256: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 artifacts saved locally.


In [37]:
# ============================================================
# PHASE 7 — G1 CELL 13
# GOLD DATASET ACCEPTANCE VALIDATION
# ============================================================

import json
import pandas as pd

# Reload saved artifacts instead of validating only memory objects.
with open(G1_GOLD_FILE, encoding="utf-8") as f:
    g1_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

g1_annotations = pd.read_csv(
    G1_ANNOTATION_FILE,
    dtype=str
)

# 1. Record count.
assert len(g1_reloaded) == 60

# 2. Unique complaint IDs.
case_ids = [
    record["case_id"]
    for record in g1_reloaded
]

assert len(case_ids) == len(set(case_ids))

# 3. Required input and expected labels.
for record in g1_reloaded:

    assert record["request"].strip()

    assert record["expected"]["product"]
    assert record["expected"]["issue"]

    assert record["source"]["dataset"] == (
        "cfpb_holdout_june_2026"
    )

# 4. Annotation template integrity.
assert len(g1_annotations) == 60
assert g1_annotations["case_id"].is_unique

assert set(g1_annotations["case_id"]) == set(case_ids)

# 5. Verify all IDs belong to the June holdout.
holdout_ids = set(
    holdout_df["Complaint ID"].astype(str)
)

assert set(case_ids).issubset(holdout_ids)

# 6. Check whether complaint identifiers are exposed
# in the existing complaint retrieval metadata.
index_ids = set()

for chunk in CASE_INDEX.chunks:

    possible_ids = [
        chunk.get("complaint_id"),
        chunk.get("Complaint ID"),
        chunk.get("case_id")
    ]

    for value in possible_ids:
        if value is not None:
            index_ids.add(str(value))

overlap = set(case_ids).intersection(index_ids)

print("=" * 65)
print("COMPLAINTIQ — G1 ACCEPTANCE VALIDATION")
print("=" * 65)

print("Gold records:", len(g1_reloaded))
print("Unique IDs:", len(set(case_ids)))
print("Valid narratives:", sum(
    bool(r["request"].strip())
    for r in g1_reloaded
))
print("Valid product labels:", sum(
    bool(r["expected"]["product"])
    for r in g1_reloaded
))
print("Valid issue labels:", sum(
    bool(r["expected"]["issue"])
    for r in g1_reloaded
))
print("Annotation rows:", len(g1_annotations))

print("\nExisting retrieval metadata ID overlap:", len(overlap))

if overlap:
    print("WARNING — Matching IDs found in retrieval metadata.")
    print("Sample matching IDs:", sorted(overlap)[:10])
else:
    print(
        "No matching complaint IDs found in the "
        "explicit metadata ID fields."
    )

print("\nArtifact checksums:")
print("Gold:", sha256_file(G1_GOLD_FILE))
print("Annotations:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G1 ACCEPTANCE VALIDATION
Gold records: 60
Unique IDs: 60
Valid narratives: 60
Valid product labels: 60
Valid issue labels: 60
Annotation rows: 60

Existing retrieval metadata ID overlap: 0
No matching complaint IDs found in the explicit metadata ID fields.

Artifact checksums:
Gold: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984
Annotations: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.


In [38]:
# ============================================================
# PHASE 7 — RETRIEVAL TOKENIZER CONSISTENCY FIX
# ============================================================

import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

# Restore the original Phase 3 tokenizer.
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

# Rebuild in-memory indexes using existing artifacts.
REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

# Validate the tokenizer.
test_tokens = tok("12 CFR 1005.11 Electronic Fund Transfers")

assert "1005.11" in test_tokens, (
    f"Unexpected tokenization: {test_tokens}"
)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("Tokenizer test:", test_tokens)
print("Regulatory index:", REG_INDEX.faiss.ntotal)
print("Complaint index:", CASE_INDEX.faiss.ntotal)

# Test regulatory retrieval.
results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

assert len(results) == 5

print("\nTop regulatory retrieval results:")
for rank, result in enumerate(results, 1):
    print(
        rank,
        result.get("chunk_id"),
        result.get("rrf")
    )

print("\nPASS — Retrieval tokenizer consistency restored.")

Tokenizer test: ['12', 'cfr', '1005.11', 'electronic', 'fund', 'transfers']
Regulatory index: 1784
Complaint index: 3000


Embedding: 100%|██████████| 1/1 [00:00<00:00, 150.02it/s]



Top regulatory retrieval results:
1 exam_manual_v2_2012_page_0700_chunk_0001 1.032522
2 exam_manual_v2_2012_page_0734_chunk_0001 1.028986
3 exam_manual_v2_2012_page_0699_chunk_0002 1.02885
4 12CFR1026_1026.13_chunk_0004 1.028405
5 exam_manual_v2_2012_page_0735_chunk_0001 1.028283

PASS — Retrieval tokenizer consistency restored.


In [39]:
# ============================================================
# PHASE 7 — G2 CELL 1
# ANALYTICS SCHEMA VALIDATION AND SQL HELPERS
# ============================================================

import json
import math
import pandas as pd
import numpy as np
from datetime import date, datetime
from decimal import Decimal

G2_GOLD_FILE = GOLD_DIR / "G2_analytics_gold.jsonl"
G2_EXPECTED_FILE = GOLD_DIR / "G2_analytics_expected.jsonl"
G2_SUMMARY_FILE = RESULT_DIR / "G2_reference_results_summary.csv"

assert "ro" in globals(), "Read-only DuckDB connection missing."
assert "guard_sql" in globals(), "SQL guard function missing."
assert "analyst_node" in globals(), "Analyst node missing."

# Inspect actual v_complaints schema.
g2_schema_df = ro.execute(
    "DESCRIBE v_complaints"
).fetchdf()

g2_columns = set(g2_schema_df["column_name"].str.lower())

print("=" * 70)
print("G2 — ANALYTICS SCHEMA")
print("=" * 70)

display(g2_schema_df)

# Resolve column aliases safely.
G2_COLUMN_CANDIDATES = {
    "complaint_id": ["complaint_id"],
    "date_received": ["date_received"],
    "product": ["product"],
    "sub_product": ["sub_product", "subproduct"],
    "issue": ["issue"],
    "sub_issue": ["sub_issue", "subissue"],
    "company": ["company"],
    "state": ["state"],
    "tags": ["tags"],
    "submitted_via": ["submitted_via"],
    "company_response": [
        "company_response",
        "company_response_to_consumer"
    ],
    "timely": [
        "timely",
        "timely_response"
    ]
}

G2_COL = {}

for logical_name, candidates in G2_COLUMN_CANDIDATES.items():

    found = next(
        (c for c in candidates if c.lower() in g2_columns),
        None
    )

    if found:
        G2_COL[logical_name] = found


required_g2_columns = [
    "complaint_id",
    "date_received",
    "product",
    "issue",
    "company",
    "state",
    "tags",
    "timely",
    "company_response"
]

missing_g2 = [
    c for c in required_g2_columns
    if c not in G2_COL
]

assert not missing_g2, (
    f"Required G2 columns unavailable: {missing_g2}"
)

# SQL identifier quoting.
def g2_qcol(logical_name):
    actual = G2_COL[logical_name]
    return '"' + actual.replace('"', '""') + '"'


# Common SQL fields.
G2_DATE = g2_qcol("date_received")
G2_PRODUCT = g2_qcol("product")
G2_ISSUE = g2_qcol("issue")
G2_COMPANY = g2_qcol("company")
G2_STATE = g2_qcol("state")
G2_TAGS = g2_qcol("tags")
G2_TIMELY = g2_qcol("timely")
G2_RESPONSE = g2_qcol("company_response")
G2_ID = g2_qcol("complaint_id")

G2_START = "DATE '2026-01-01'"
G2_END = "DATE '2026-07-01'"

G2_WINDOW = f"""
{G2_DATE} >= {G2_START}
AND {G2_DATE} < {G2_END}
"""

# Only two in-scope products.
G2_PRODUCTS = (
    "'Credit card', "
    "'Checking or savings account'"
)

# JSON-safe conversion.
def g2_json_safe(value):

    if value is None:
        return None

    if isinstance(value, dict):
        return {
            str(k): g2_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            g2_json_safe(v)
            for v in value
        ]

    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return None if np.isnan(value) else float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, Decimal):
        return float(value)

    if isinstance(value, float) and not math.isfinite(value):
        return None

    return value


print("\nResolved columns:")
for logical, actual in G2_COL.items():
    print(f"{logical:20} -> {actual}")

print("\nEvaluation period: January 1 – June 30, 2026")
print("Approved source: v_complaints")
print("Connection: read-only")

print("\nPASS — G2 schema and SQL helpers initialized.")

G2 — ANALYTICS SCHEMA


,column_name,column_type,null,key,default,extra
0,date_received,DATE,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



Resolved columns:
complaint_id         -> complaint_id
date_received        -> date_received
product              -> product
sub_product          -> subproduct
issue                -> issue
sub_issue            -> subissue
company              -> company
state                -> state
tags                 -> tags
submitted_via        -> submitted_via
company_response     -> company_response_to_consumer
timely               -> timely_response

Evaluation period: January 1 – June 30, 2026
Approved source: v_complaints
Connection: read-only

PASS — G2 schema and SQL helpers initialized.


In [40]:
# ============================================================
# PHASE 7 — G2 CELL 2
# DEFINE 20 ANALYTICS GOLD QUESTIONS
# ============================================================

g2_questions = []

def add_g2_question(
    question_id,
    question,
    category,
    reference_sql=None,
    answerable=True,
    refusal_reason=None
):
    g2_questions.append({
        "question_id": question_id,
        "question": question,
        "category": category,
        "answerable": answerable,
        "reference_sql": reference_sql,
        "refusal_reason": refusal_reason
    })


# ------------------------------------------------------------
# G2-01 — Top five credit card issues in June
# Date filter + GROUP BY + ORDER BY
# ------------------------------------------------------------

add_g2_question(
    "G2-01",
    "What were the five most common issues for credit card complaints received in June 2026, with counts?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_count DESC, issue ASC
    LIMIT 5
    """
)


# ------------------------------------------------------------
# G2-02 — Monthly Older American complaints
# ------------------------------------------------------------

add_g2_question(
    "G2-02",
    "Show monthly complaint counts for Checking or savings account complaints tagged Older American from January through June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Checking or savings account'
      AND {G2_DATE} >= DATE '2026-01-01'
      AND {G2_DATE} < DATE '2026-07-01'
      AND {G2_TAGS} ILIKE '%Older American%'
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-03 — Lowest timely response rate
# ------------------------------------------------------------

add_g2_question(
    "G2-03",
    "Among companies with at least 200 complaints from January through June 2026, which five companies had the lowest timely response rate?",
    "group_order",
    f"""
    SELECT
        {G2_COMPANY} AS company,
        COUNT(*) AS complaint_count,
        COUNT(*) FILTER (
            WHERE {G2_TIMELY} = 'Yes'
        ) AS timely_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_TIMELY} = 'Yes'
            ) / COUNT(*),
            2
        ) AS timely_response_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    HAVING COUNT(*) >= 200
    ORDER BY timely_response_pct ASC, complaint_count DESC
    LIMIT 5
    """
)


# ------------------------------------------------------------
# G2-04 — Monetary relief share by product
# ------------------------------------------------------------

add_g2_question(
    "G2-04",
    "What percentage of complaints received monetary relief, broken down by product, from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
        ) AS monetary_relief_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
            ) / COUNT(*),
            2
        ) AS monetary_relief_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY monetary_relief_pct DESC
    """
)


# ------------------------------------------------------------
# G2-05 — Fastest growing purchase sub-issue
# ------------------------------------------------------------

assert "sub_issue" in G2_COL, (
    "G2-05 requires the sub_issue column."
)

G2_SUBISSUE = g2_qcol("sub_issue")

add_g2_question(
    "G2-05",
    "Which sub-issue under Problem with a purchase shown on your statement had the largest increase in complaint count from Q1 to Q2 2026?",
    "date_filter_group_order",
    f"""
    WITH quarterly AS (
        SELECT
            {G2_SUBISSUE} AS sub_issue,
            COUNT(*) FILTER (
                WHERE {G2_DATE} < DATE '2026-04-01'
            ) AS q1_count,
            COUNT(*) FILTER (
                WHERE {G2_DATE} >= DATE '2026-04-01'
            ) AS q2_count
        FROM v_complaints
        WHERE {G2_PRODUCT} = 'Credit card'
          AND {G2_ISSUE} =
              'Problem with a purchase shown on your statement'
          AND {G2_DATE} >= DATE '2026-01-01'
          AND {G2_DATE} < DATE '2026-07-01'
          AND {G2_SUBISSUE} IS NOT NULL
        GROUP BY 1
    )
    SELECT
        sub_issue,
        q1_count,
        q2_count,
        q2_count - q1_count AS count_change
    FROM quarterly
    ORDER BY count_change DESC, sub_issue ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-06 — Monthly complaints by product
# ------------------------------------------------------------

add_g2_question(
    "G2-06",
    "Show monthly complaint counts by product for January through June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        {G2_PRODUCT} AS product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY complaint_month, product
    """
)


# ------------------------------------------------------------
# G2-07 — Credit card monthly trend
# ------------------------------------------------------------

add_g2_question(
    "G2-07",
    "How many credit card complaints were received in each month from January through June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-08 — Top ten companies by complaint volume
# ------------------------------------------------------------

add_g2_question(
    "G2-08",
    "Which ten companies received the most complaints across the two supported products from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_COMPANY} AS company,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1
    ORDER BY complaint_count DESC, company ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-09 — Timely response rate by product
# ------------------------------------------------------------

add_g2_question(
    "G2-09",
    "What was the timely response rate for each supported product from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_TIMELY} = 'Yes'
        ) AS timely_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_TIMELY} = 'Yes'
            ) / COUNT(*),
            2
        ) AS timely_response_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1
    ORDER BY product
    """
)


# ------------------------------------------------------------
# G2-10 — Top checking account issues in Q2
# ------------------------------------------------------------

add_g2_question(
    "G2-10",
    "What were the ten most common issues for Checking or savings account complaints in Q2 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Checking or savings account'
      AND {G2_DATE} >= DATE '2026-04-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_count DESC, issue ASC
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-11 — Complaints by submission channel
# ------------------------------------------------------------

assert "submitted_via" in G2_COL, (
    "G2-11 requires submitted_via."
)

G2_CHANNEL = g2_qcol("submitted_via")

add_g2_question(
    "G2-11",
    "How many complaints were submitted through each channel from January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_CHANNEL} AS submitted_via,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_count DESC, submitted_via
    """
)


# ------------------------------------------------------------
# G2-12 — Top states for credit card complaints in June
# ------------------------------------------------------------

add_g2_question(
    "G2-12",
    "Which ten states had the highest number of credit card complaints in June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_STATE} AS state,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
      AND {G2_STATE} IS NOT NULL
    GROUP BY 1
    ORDER BY complaint_count DESC, state
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-13 — Quarterly complaint counts by product
# ------------------------------------------------------------

add_g2_question(
    "G2-13",
    "Compare Q1 and Q2 2026 complaint counts for each supported product.",
    "date_filter_group_order",
    f"""
    SELECT
        CASE
            WHEN {G2_DATE} < DATE '2026-04-01' THEN 'Q1 2026'
            ELSE 'Q2 2026'
        END AS quarter,
        {G2_PRODUCT} AS product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY quarter, product
    """
)


# ------------------------------------------------------------
# G2-14 — Sub-product distribution
# ------------------------------------------------------------

assert "sub_product" in G2_COL, (
    "G2-14 requires sub_product."
)

G2_SUBPRODUCT = g2_qcol("sub_product")

add_g2_question(
    "G2-14",
    "What are the complaint counts by sub-product for each supported product during January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_PRODUCT} AS product,
        {G2_SUBPRODUCT} AS sub_product,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
    GROUP BY 1, 2
    ORDER BY product, complaint_count DESC
    LIMIT 200
    """
)


# ------------------------------------------------------------
# G2-15 — Daily June complaint volume
# ------------------------------------------------------------

add_g2_question(
    "G2-15",
    "Show the daily complaint counts for both supported products during June 2026.",
    "date_filter_group_order",
    f"""
    SELECT
        {G2_DATE} AS complaint_date,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
      AND {G2_PRODUCT} IN ({G2_PRODUCTS})
      AND {G2_DATE} >= DATE '2026-06-01'
      AND {G2_DATE} < DATE '2026-07-01'
    GROUP BY 1
    ORDER BY complaint_date
    """
)


# ------------------------------------------------------------
# G2-16 — Timely response by submission channel
# ------------------------------------------------------------

add_g2_question(
    "G2-16",
    "Compare timely and untimely response counts by submission channel from January through June 2026.",
    "group_order",
    f"""
    SELECT
        {G2_CHANNEL} AS submitted_via,
        {G2_TIMELY} AS timely_response,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1, 2
    ORDER BY submitted_via, timely_response
    """
)


# ------------------------------------------------------------
# G2-17 — Most common credit card issues
# ------------------------------------------------------------

add_g2_question(
    "G2-17",
    "What were the ten most common credit card complaint issues during January through June 2026?",
    "group_order",
    f"""
    SELECT
        {G2_ISSUE} AS issue,
        COUNT(*) AS complaint_count
    FROM v_complaints
    WHERE {G2_PRODUCT} = 'Credit card'
      AND {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_count DESC, issue
    LIMIT 10
    """
)


# ------------------------------------------------------------
# G2-18 — Monthly monetary relief rate
# ------------------------------------------------------------

add_g2_question(
    "G2-18",
    "How did the monthly percentage of complaints receiving monetary relief change from January through June 2026?",
    "date_filter_group_order",
    f"""
    SELECT
        DATE_TRUNC('month', {G2_DATE})::DATE AS complaint_month,
        COUNT(*) AS total_complaints,
        COUNT(*) FILTER (
            WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
        ) AS monetary_relief_count,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE {G2_RESPONSE} ILIKE '%monetary relief%'
            ) / COUNT(*),
            2
        ) AS monetary_relief_pct
    FROM v_complaints
    WHERE {G2_WINDOW}
    GROUP BY 1
    ORDER BY complaint_month
    """
)


# ------------------------------------------------------------
# G2-19 — Unanswerable: per-capita complaints
# ------------------------------------------------------------

add_g2_question(
    "G2-19",
    "Which state has the highest number of complaints per 100,000 residents during January through June 2026?",
    "unanswerable",
    answerable=False,
    refusal_reason=(
        "The v_complaints view contains complaint counts and state "
        "but no state population or demographic denominator. "
        "Per-capita rates cannot be calculated from this dataset."
    )
)


# ------------------------------------------------------------
# G2-20 — Unanswerable: consumer income
# ------------------------------------------------------------

add_g2_question(
    "G2-20",
    "What is the average annual household income of consumers who submitted complaints, broken down by product?",
    "unanswerable",
    answerable=False,
    refusal_reason=(
        "The v_complaints view does not contain consumer household "
        "income. The requested average cannot be calculated."
    )
)


# ------------------------------------------------------------
# Validate question definitions
# ------------------------------------------------------------

assert len(g2_questions) == 20

assert len([
    q for q in g2_questions
    if q["answerable"]
]) == 18

assert len([
    q for q in g2_questions
    if not q["answerable"]
]) == 2

assert len([
    q for q in g2_questions
    if q["category"] == "unanswerable"
]) == 2

assert all(
    q["reference_sql"]
    for q in g2_questions
    if q["answerable"]
)

print("=" * 70)
print("G2 — GOLD QUESTION INVENTORY")
print("=" * 70)

display(pd.DataFrame([
    {
        "question_id": q["question_id"],
        "category": q["category"],
        "answerable": q["answerable"],
        "question": q["question"]
    }
    for q in g2_questions
]))

print("\nTotal questions:", len(g2_questions))
print("Answerable:", sum(q["answerable"] for q in g2_questions))
print("Unanswerable:", sum(not q["answerable"] for q in g2_questions))

print("\nPASS — 20 G2 gold questions defined.")

G2 — GOLD QUESTION INVENTORY


,question_id,category,answerable,question
0,G2-01,date_filter_group_order,True,What were the five most common issues for cred...
1,G2-02,date_filter_group_order,True,Show monthly complaint counts for Checking or ...
2,G2-03,group_order,True,Among companies with at least 200 complaints f...
3,G2-04,group_order,True,What percentage of complaints received monetar...
4,G2-05,date_filter_group_order,True,Which sub-issue under Problem with a purchase ...
5,G2-06,date_filter_group_order,True,Show monthly complaint counts by product for J...
6,G2-07,date_filter_group_order,True,How many credit card complaints were received ...
7,G2-08,group_order,True,Which ten companies received the most complain...
8,G2-09,group_order,True,What was the timely response rate for each sup...
9,G2-10,date_filter_group_order,True,What were the ten most common issues for Check...



Total questions: 20
Answerable: 18
Unanswerable: 2

PASS — 20 G2 gold questions defined.


In [41]:
# ============================================================
# PHASE 7 — G2 CELL 3
# EXECUTE REFERENCE SQL AND SAVE EXPECTED RESULTS
# ============================================================

g2_expected_records = []
g2_execution_summary = []

for item in g2_questions:

    question_id = item["question_id"]

    print("\n" + "=" * 70)
    print(question_id, "|", item["category"])
    print("=" * 70)

    if not item["answerable"]:

        g2_expected_records.append({
            "question_id": question_id,
            "answerable": False,
            "expected_rows": None,
            "expected_row_count": None,
            "refusal_reason": item["refusal_reason"]
        })

        g2_execution_summary.append({
            "question_id": question_id,
            "answerable": False,
            "execution_status": "EXPECTED_REFUSAL",
            "expected_row_count": None
        })

        print("Expected outcome: REFUSAL")
        print("Reason:", item["refusal_reason"])

        continue

    sql = item["reference_sql"]

    # Validate every reference query through the same SQL guard.
    safe_sql = guard_sql(sql)

    # Execute on the read-only database connection.
    result_df = ro.execute(safe_sql).fetchdf()

    rows = [
        {
            str(k): g2_json_safe(v)
            for k, v in row.items()
        }
        for row in result_df.to_dict("records")
    ]

    g2_expected_records.append({
        "question_id": question_id,
        "answerable": True,
        "expected_rows": rows,
        "expected_row_count": len(rows),
        "refusal_reason": None
    })

    g2_execution_summary.append({
        "question_id": question_id,
        "answerable": True,
        "execution_status": "PASS",
        "expected_row_count": len(rows)
    })

    print("Reference SQL: PASS")
    print("Expected rows:", len(rows))

    if rows:
        display(result_df.head(10))


# ------------------------------------------------------------
# Save reference results
# ------------------------------------------------------------

with open(G2_EXPECTED_FILE, "w", encoding="utf-8") as f:

    for record in g2_expected_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# Save question definitions.
with open(G2_GOLD_FILE, "w", encoding="utf-8") as f:

    for item in g2_questions:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            ) + "\n"
        )


g2_summary_df = pd.DataFrame(g2_execution_summary)

g2_summary_df.to_csv(
    G2_SUMMARY_FILE,
    index=False
)

print("\n" + "=" * 70)
print("G2 — REFERENCE SQL EXECUTION SUMMARY")
print("=" * 70)

display(g2_summary_df)

answerable_passes = sum(
    x["execution_status"] == "PASS"
    for x in g2_execution_summary
)

refusal_cases = sum(
    x["execution_status"] == "EXPECTED_REFUSAL"
    for x in g2_execution_summary
)

print("Answerable reference queries passed:", answerable_passes)
print("Expected refusal cases:", refusal_cases)

assert answerable_passes == 18
assert refusal_cases == 2
assert len(g2_expected_records) == 20

print("\nPASS — All G2 reference queries executed and saved.")


G2-01 | date_filter_group_order
Reference SQL: PASS
Expected rows: 5


,issue,complaint_count
0,Problem with a purchase shown on your statement,995
1,"Other features, terms, or problems",359
2,Fees or interest,345
3,Getting a credit card,256
4,Closing your account,222



G2-02 | date_filter_group_order
Reference SQL: PASS
Expected rows: 6


,complaint_month,complaint_count
0,2026-01-01,280
1,2026-02-01,169
2,2026-03-01,290
3,2026-04-01,273
4,2026-05-01,254
5,2026-06-01,304



G2-03 | group_order
Reference SQL: PASS
Expected rows: 5


,company,complaint_count,timely_count,timely_response_pct
0,FinCo Services Inc DBA Current,237,189,79.75
1,"Bilt Technologies, Inc.",213,173,81.22
2,SYNCHRONY FINANCIAL,2083,1948,93.52
3,ROBINHOOD MARKETS INC.,240,237,98.75
4,"BANK OF AMERICA, NATIONAL ASSOCIATION",3090,3054,98.83


... (29 outputs left out)

,complaint_month,total_complaints,monetary_relief_count,monetary_relief_pct
0,2026-01-01,7580,1691,22.31
1,2026-02-01,3962,894,22.56
2,2026-03-01,6812,1499,22.01
3,2026-04-01,7380,1625,22.02
4,2026-05-01,6079,1362,22.41
5,2026-06-01,6110,1411,23.09



G2-19 | unanswerable
Expected outcome: REFUSAL
Reason: The v_complaints view contains complaint counts and state but no state population or demographic denominator. Per-capita rates cannot be calculated from this dataset.

G2-20 | unanswerable
Expected outcome: REFUSAL
Reason: The v_complaints view does not contain consumer household income. The requested average cannot be calculated.

G2 — REFERENCE SQL EXECUTION SUMMARY


,question_id,answerable,execution_status,expected_row_count
0,G2-01,True,PASS,5.0
1,G2-02,True,PASS,6.0
2,G2-03,True,PASS,5.0
3,G2-04,True,PASS,2.0
4,G2-05,True,PASS,3.0
5,G2-06,True,PASS,12.0
6,G2-07,True,PASS,6.0
7,G2-08,True,PASS,10.0
8,G2-09,True,PASS,2.0
9,G2-10,True,PASS,10.0


Answerable reference queries passed: 18
Expected refusal cases: 2

PASS — All G2 reference queries executed and saved.


In [42]:
# ============================================================
# PHASE 7 — G2 CELL 4
# GOLD DATASET ACCEPTANCE VALIDATION
# ============================================================

import json
import hashlib
import pandas as pd

# Reload the saved question definitions.
with open(G2_GOLD_FILE, encoding="utf-8") as f:
    g2_gold_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Reload expected result sets.
with open(G2_EXPECTED_FILE, encoding="utf-8") as f:
    g2_expected_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Reload summary.
g2_summary_reloaded = pd.read_csv(
    G2_SUMMARY_FILE
)

# 1. Count validation.
assert len(g2_gold_reloaded) == 20
assert len(g2_expected_reloaded) == 20

# 2. Unique question IDs.
g2_ids = [
    x["question_id"]
    for x in g2_gold_reloaded
]

assert len(g2_ids) == len(set(g2_ids))

# 3. Answerable and unanswerable counts.
answerable_items = [
    x for x in g2_gold_reloaded
    if x["answerable"]
]

refusal_items = [
    x for x in g2_gold_reloaded
    if not x["answerable"]
]

assert len(answerable_items) == 18
assert len(refusal_items) == 2

# 4. Every answerable question has reference SQL.
for item in answerable_items:

    assert item["reference_sql"].strip()

    parsed = sqlglot.parse(
        item["reference_sql"],
        read="duckdb"
    )

    assert len(parsed) == 1
    assert isinstance(parsed[0], sqlglot.exp.Select)

    safe = guard_sql(item["reference_sql"])

    assert "v_complaints" in safe.lower()

# 5. Every refusal question has a reason.
for item in refusal_items:

    assert item["refusal_reason"]
    assert item["reference_sql"] is None

# 6. Expected result records.
expected_by_id = {
    x["question_id"]: x
    for x in g2_expected_reloaded
}

for item in answerable_items:

    expected = expected_by_id[item["question_id"]]

    assert expected["answerable"] is True
    assert expected["expected_rows"] is not None
    assert expected["expected_row_count"] == len(
        expected["expected_rows"]
    )

for item in refusal_items:

    expected = expected_by_id[item["question_id"]]

    assert expected["answerable"] is False
    assert expected["refusal_reason"]

# 7. All answerable reference executions passed.
assert (
    g2_summary_reloaded["execution_status"] == "PASS"
).sum() == 18

assert (
    g2_summary_reloaded["execution_status"] == "EXPECTED_REFUSAL"
).sum() == 2


# 8. SHA256 checksums.
def g2_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


print("=" * 70)
print("COMPLAINTIQ — G2 ACCEPTANCE VALIDATION")
print("=" * 70)

print("Total gold questions:", len(g2_gold_reloaded))
print("Unique question IDs:", len(set(g2_ids)))
print("Answerable questions:", len(answerable_items))
print("Unanswerable questions:", len(refusal_items))

print(
    "Reference SQL execution passes:",
    int((
        g2_summary_reloaded["execution_status"] == "PASS"
    ).sum())
)

print(
    "Expected refusal cases:",
    int((
        g2_summary_reloaded["execution_status"] == "EXPECTED_REFUSAL"
    ).sum())
)

print("\nSaved artifacts:")

for path in [
    G2_GOLD_FILE,
    G2_EXPECTED_FILE,
    G2_SUMMARY_FILE
]:

    assert path.exists()

    print("\n", path)
    print("Size:", path.stat().st_size, "bytes")
    print("SHA256:", g2_sha256(path))


print("\nPASS — G2 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G2 ACCEPTANCE VALIDATION
Total gold questions: 20
Unique question IDs: 20
Answerable questions: 18
Unanswerable questions: 2
Reference SQL execution passes: 18
Expected refusal cases: 2

Saved artifacts:

 /home/sagemaker-user/complaintiq/eval/gold/G2_analytics_gold.jsonl
Size: 13141 bytes
SHA256: 21f268c50803d8286db93377dbf5d2d342e3b63fa302216d2e33239ed1bd133b

 /home/sagemaker-user/complaintiq/eval/gold/G2_analytics_expected.jsonl
Size: 12950 bytes
SHA256: 9e57b566bc49d056d1b6e818b78037eb62af0553f3d3252c76b1aa830a87bbac

 /home/sagemaker-user/complaintiq/eval/results/G2_reference_results_summary.csv
Size: 485 bytes
SHA256: bc83b3ca5dff7a962388d1b0364d9b142480d8231f288de0090b5cce891330f9

PASS — G2 GOLD DATASET STRUCTURAL ACCEPTANCE.


In [43]:
# ============================================================
# PHASE 7 — G3 CELL 43
# REGULATORY CORPUS INSPECTION
# ============================================================

import re
import json
import pandas as pd
from collections import Counter

assert "REG_INDEX" in globals()
assert len(REG_INDEX.chunks) == 1784

g3_chunks = REG_INDEX.chunks

print("=" * 70)
print("G3 — REGULATORY CORPUS INSPECTION")
print("=" * 70)

print("Total indexed chunks:", len(g3_chunks))

# Inspect metadata structure.
print("\nFirst chunk metadata:")
print(json.dumps(g3_chunks[0], indent=2, ensure_ascii=False)[:3500])

# Source distribution.
source_counts = Counter(
    str(c.get("source_type", "UNKNOWN"))
    for c in g3_chunks
)

print("\nSource type distribution:")
display(
    pd.DataFrame(
        source_counts.items(),
        columns=["source_type", "chunk_count"]
    )
)

# Source file distribution.
file_counts = Counter(
    str(c.get("source_file", "UNKNOWN"))
    for c in g3_chunks
)

print("\nSource file distribution:")
display(
    pd.DataFrame(
        file_counts.items(),
        columns=["source_file", "chunk_count"]
    ).sort_values("chunk_count", ascending=False)
)

# Extract CFR references from metadata and text.
cfr_pattern = re.compile(
    r"12\s*CFR\s*(1005|1026|1022|1006|1030)\.(\d+[a-z]?)",
    re.I
)

g3_cfr_counts = Counter()

for chunk in g3_chunks:

    searchable = " ".join([
        str(chunk.get("chunk_id", "")),
        str(chunk.get("source_id", "")),
        str(chunk.get("citation", "")),
        str(chunk.get("title", "")),
        str(chunk.get("text", ""))
    ])

    for part, section in cfr_pattern.findall(searchable):
        g3_cfr_counts[f"12 CFR {part}.{section}"] += 1


print("\nCFR sections detected in corpus:")
display(
    pd.DataFrame(
        g3_cfr_counts.items(),
        columns=["cfr_section", "matching_chunks"]
    ).sort_values("cfr_section")
)

assert len(g3_cfr_counts) > 0

print("\nPASS — Regulatory corpus metadata inspected.")

G3 — REGULATORY CORPUS INSPECTION
Total indexed chunks: 1784

First chunk metadata:
{
  "chunk_id": "12CFR1002_1002.1_chunk_0001",
  "source_type": "ecfr",
  "source_id": "12CFR1002_1002.1",
  "source_file": "title_12_part_1002_2026-11-03.xml",
  "citation": "12 CFR Part 1002, Section 1002.1",
  "title": "§ 1002.1 Authority, scope and purpose.",
  "chunk_number": 1,
  "text": "(a) Authority and scope. This part, known as Regulation B, is issued by the Bureau of Consumer Financial Protection (Bureau) pursuant to title VII (Equal Credit Opportunity Act) of the Consumer Credit Protection Act, as amended (15 U.S.C. 1601 et seq. ). Except as otherwise provided herein, this subpart applies to all persons who are creditors, as defined in § 1002.2(l), other than a person excluded from coverage of this part by section 1029 of the Consumer Financial Protection Act of 2010, title X of the Dodd-Frank Wall Street Reform and Consumer Protection Act, Public Law 111-203, 124 Stat. 1376. Information co

,source_type,chunk_count
0,ecfr,646
1,pdf,1138



Source file distribution:


,source_file,chunk_count
6,exam_manual_v2_2012.pdf,1029
4,title_12_part_1026_2026-11-03.xml,358
1,title_12_part_1005_2026-11-03.xml,100
3,title_12_part_1022_2026-11-03.xml,99
0,title_12_part_1002_2026-11-03.xml,55
9,sh_issue22_2020-09.pdf,37
2,title_12_part_1006_2026-11-03.xml,31
10,sh_issue23_2021-01.pdf,30
7,sh_issue17_2018-09.pdf,22
8,sh_issue18_2019-03.pdf,20



CFR sections detected in corpus:


,cfr_section,matching_chunks
68,12 CFR 1005.10,33
15,12 CFR 1005.11,38
14,12 CFR 1005.12,7
69,12 CFR 1005.13,5
74,12 CFR 1005.14,6
...,...,...
85,12 CFR 1030.5,41
82,12 CFR 1030.6,31
84,12 CFR 1030.7,31
80,12 CFR 1030.8,103



PASS — Regulatory corpus metadata inspected.


In [ ]:
# ============================================================
# PHASE 7 — G3 CELL 44
# DEFINE REGULATORY GOLD QUESTIONS
# ============================================================

g3_questions = []

def add_g3_question(
    question_id,
    question,
    regulation,
    expected_sections=None,
    expected_source=None,
    answerable=True,
    abstention_reason=None
):

    g3_questions.append({
        "question_id": question_id,
        "question": question,
        "regulation": regulation,
        "answerable": answerable,
        "expected_sections": expected_sections or [],
        "expected_source": expected_source,
        "gold_chunk_ids": [],
        "abstention_reason": abstention_reason
    })


# ------------------------------------------------------------
# REGULATION E — 12 CFR PART 1005
# ------------------------------------------------------------

add_g3_question(
    "G3-01",
    "After a consumer notifies a financial institution of an electronic fund transfer error, what investigation and resolution steps are required, and within what time limits?",
    "Regulation E",
    ["12 CFR 1005.11"]
)

add_g3_question(
    "G3-02",
    "What liability limits apply when an access device is lost or stolen and an unauthorized electronic fund transfer occurs?",
    "Regulation E",
    ["12 CFR 1005.6"]
)

add_g3_question(
    "G3-03",
    "What initial disclosures must a financial institution provide for an electronic fund transfer service?",
    "Regulation E",
    ["12 CFR 1005.7"]
)

add_g3_question(
    "G3-04",
    "What are the requirements for providing periodic statements for accounts subject to Regulation E?",
    "Regulation E",
    ["12 CFR 1005.9"]
)


# ------------------------------------------------------------
# REGULATION Z — 12 CFR PART 1026
# ------------------------------------------------------------

add_g3_question(
    "G3-05",
    "What constitutes a billing error on an open-end credit account, and what must a creditor do after receiving a proper billing-error notice?",
    "Regulation Z",
    ["12 CFR 1026.13"]
)

add_g3_question(
    "G3-06",
    "What limits apply to a consumer's liability for unauthorized use of a credit card?",
    "Regulation Z",
    ["12 CFR 1026.12"]
)

add_g3_question(
    "G3-07",
    "What disclosures must be provided before opening an open-end credit account?",
    "Regulation Z",
    ["12 CFR 1026.6"]
)


# ------------------------------------------------------------
# REGULATION V — 12 CFR PART 1022
# ------------------------------------------------------------

add_g3_question(
    "G3-08",
    "What duties does a furnisher have after receiving a direct dispute from a consumer about information furnished to a consumer reporting agency?",
    "Regulation V",
    ["12 CFR 1022.43"]
)

add_g3_question(
    "G3-09",
    "What policies and procedures must furnishers maintain to ensure the accuracy and integrity of information they report to consumer reporting agencies?",
    "Regulation V",
    ["12 CFR 1022.42"]
)

add_g3_question(
    "G3-10",
    "What types of direct disputes are excluded from the furnisher investigation requirements, and what happens when a dispute is frivolous or irrelevant?",
    "Regulation V",
    ["12 CFR 1022.43"]
)


# ------------------------------------------------------------
# REGULATION F — 12 CFR PART 1006
# ------------------------------------------------------------

add_g3_question(
    "G3-11",
    "What restrictions apply to the frequency and timing of telephone calls by a debt collector to a consumer?",
    "Regulation F",
    ["12 CFR 1006.14"]
)

add_g3_question(
    "G3-12",
    "When and how may a debt collector communicate with a consumer, and what restrictions apply to communication at inconvenient times or places?",
    "Regulation F",
    ["12 CFR 1006.6"]
)

add_g3_question(
    "G3-13",
    "What information must a debt collector provide in a validation notice, and when must it be sent?",
    "Regulation F",
    ["12 CFR 1006.34"]
)


# ------------------------------------------------------------
# REGULATION DD — 12 CFR PART 1030
# ------------------------------------------------------------

add_g3_question(
    "G3-14",
    "What account-opening disclosures must a depository institution provide under Regulation DD?",
    "Regulation DD",
    ["12 CFR 1030.4"]
)

add_g3_question(
    "G3-15",
    "What rules govern the payment of interest on interest-bearing deposit accounts?",
    "Regulation DD",
    ["12 CFR 1030.7"]
)

add_g3_question(
    "G3-16",
    "What requirements apply to the calculation of interest on deposit accounts?",
    "Regulation DD",
    ["12 CFR 1030.7"]
)


# ------------------------------------------------------------
# SUPERVISORY DOCUMENTS
# ------------------------------------------------------------

add_g3_question(
    "G3-17",
    "What consumer-protection or compliance findings concerning deposit accounts are described in the CFPB Supervisory Highlights documents included in the corpus?",
    "CFPB Supervisory Highlights",
    [],
    expected_source="supervisory_highlights"
)


# ------------------------------------------------------------
# THREE QUESTIONS THAT MUST BE REFUSED
# ------------------------------------------------------------

add_g3_question(
    "G3-18",
    "What is Harbourline Bank's internal policy on cryptocurrency custody, and which internal approval committee authorizes it?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains no Harbourline internal cryptocurrency "
        "custody policy or internal committee records."
    )
)

add_g3_question(
    "G3-19",
    "What is the exact contractual refund processing time promised by Harbourline Bank to its customers?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains regulations and supervisory documents, "
        "not Harbourline's internal customer contracts or service-level commitments."
    )
)

add_g3_question(
    "G3-20",
    "Which named Harbourline employee approved a specific customer's refund last week?",
    "Not in corpus",
    answerable=False,
    abstention_reason=(
        "The corpus contains no Harbourline employee records or internal "
        "customer-level approval history."
    )
)


# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

assert len(g3_questions) == 20

assert sum(q["answerable"] for q in g3_questions) == 17

assert sum(not q["answerable"] for q in g3_questions) == 3

assert len({
    q["question_id"] for q in g3_questions
}) == 20

print("=" * 70)
print("G3 — QUESTION INVENTORY")
print("=" * 70)

display(pd.DataFrame([
    {
        "question_id": q["question_id"],
        "regulation": q["regulation"],
        "answerable": q["answerable"],
        "expected_sections": ", ".join(q["expected_sections"]),
        "question": q["question"]
    }
    for q in g3_questions
]))

print("\nTotal questions:", len(g3_questions))
print("Answerable:", sum(q["answerable"] for q in g3_questions))
print("Abstention:", sum(not q["answerable"] for q in g3_questions))

print("\nPASS — 20 G3 questions defined.")

In [ ]:
# ============================================================
# PHASE 7 — G3 CELL 45
# RESOLVE GOLD SECTION IDS AND CHUNK IDS
# ============================================================

import re
import pandas as pd
from collections import defaultdict

def g3_chunk_searchable(chunk):

    return " ".join([
        str(chunk.get("chunk_id", "")),
        str(chunk.get("source_id", "")),
        str(chunk.get("citation", "")),
        str(chunk.get("title", "")),
        str(chunk.get("source_file", "")),
        str(chunk.get("text", ""))
    ])


def g3_normalize_cfr(section):

    return re.sub(
        r"\s+",
        "",
        str(section).lower()
    ).replace("12cfr", "")


def g3_find_section_chunks(section):

    target = g3_normalize_cfr(section)

    matches = []

    for chunk in g3_chunks:

        searchable = g3_chunk_searchable(chunk)

        normalized = re.sub(
            r"[^a-z0-9.]",
            "",
            searchable.lower()
        )

        if target in normalized:
            matches.append(chunk)

    return matches


g3_resolution_rows = []

for item in g3_questions:

    if not item["answerable"]:
        continue

    if item["expected_sections"]:

        matched_chunks = []

        for section in item["expected_sections"]:

            section_chunks = g3_find_section_chunks(section)

            for chunk in section_chunks:
                if chunk not in matched_chunks:
                    matched_chunks.append(chunk)

        item["gold_chunk_ids"] = [
            str(c["chunk_id"])
            for c in matched_chunks
            if c.get("chunk_id") is not None
        ]

        g3_resolution_rows.append({
            "question_id": item["question_id"],
            "expected_sections": ", ".join(
                item["expected_sections"]
            ),
            "matching_chunks": len(matched_chunks),
            "gold_chunk_ids": ", ".join(
                item["gold_chunk_ids"][:8]
            ),
            "status": (
                "PASS" if item["gold_chunk_ids"] else "MISSING"
            )
        })


# ------------------------------------------------------------
# Supervisory Highlights retrieval candidates
# ------------------------------------------------------------

g3_supervisory_question = next(
    q for q in g3_questions
    if q["question_id"] == "G3-17"
)

g3_supervisory_candidates = REG_INDEX.search(
    g3_supervisory_question["question"],
    k=12
)

g3_supervisory_rows = []

for rank, chunk in enumerate(g3_supervisory_candidates, 1):

    g3_supervisory_rows.append({
        "rank": rank,
        "chunk_id": chunk.get("chunk_id"),
        "source_file": chunk.get("source_file"),
        "citation": chunk.get("citation"),
        "title": chunk.get("title"),
        "text_preview": str(chunk.get("text", ""))[:700]
    })


print("=" * 70)
print("G3 — CFR SECTION RESOLUTION")
print("=" * 70)

g3_resolution_df = pd.DataFrame(g3_resolution_rows)
display(g3_resolution_df)

print("\nSupervisory Highlights retrieval candidates:")
display(pd.DataFrame(g3_supervisory_rows))

# Required sections must exist.
missing_sections = g3_resolution_df[
    g3_resolution_df["status"] != "PASS"
]

if not missing_sections.empty:

    print("\nREVIEW REQUIRED — Missing expected CFR sections:")
    display(missing_sections)

else:

    print("\nPASS — All expected CFR sections resolved.")


# Supervisory answer needs manual source review.
print(
    "\nREVIEW REQUIRED — For G3-17, inspect the candidate text "
    "and select only chunks that actually support the answer."
)

print("\nNo LLM-generated labels were used.")

In [ ]:
# ============================================================
# PHASE 7 — G3 CELL 46
# SAVE GOLD DATASET AND ACCEPTANCE VALIDATION
# ============================================================

import json
import hashlib
import pandas as pd

G3_GOLD_FILE = GOLD_DIR / "G3_regulatory_gold.jsonl"
G3_CANDIDATES_FILE = GOLD_DIR / "G3_supervisory_candidates.jsonl"
G3_SUMMARY_FILE = RESULTS_DIR / "G3_gold_resolution_summary.csv"

# ------------------------------------------------------------
# 1. Assign status to every question
# ------------------------------------------------------------

for item in g3_questions:

    if not item["answerable"]:

        item["gold_status"] = "ABSTENTION_GOLD"

    elif item["question_id"] == "G3-17":

        item["gold_status"] = "PENDING_MANUAL_REVIEW"

    elif item["gold_chunk_ids"]:

        item["gold_status"] = "SECTION_RESOLVED"

    else:

        item["gold_status"] = "MISSING_GOLD_SECTION"


# ------------------------------------------------------------
# 2. Save the 20 question definitions
# ------------------------------------------------------------

with open(G3_GOLD_FILE, "w", encoding="utf-8") as f:

    for item in g3_questions:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------------------------
# 3. Save supervisory retrieval candidates separately
# ------------------------------------------------------------

with open(G3_CANDIDATES_FILE, "w", encoding="utf-8") as f:

    for candidate in g3_supervisory_rows:

        f.write(
            json.dumps(
                candidate,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------------------------
# 4. Create resolution summary
# ------------------------------------------------------------

g3_summary_rows = []

for item in g3_questions:

    g3_summary_rows.append({
        "question_id": item["question_id"],
        "regulation": item["regulation"],
        "answerable": item["answerable"],
        "expected_sections": ", ".join(
            item["expected_sections"]
        ),
        "gold_chunk_count": len(item["gold_chunk_ids"]),
        "gold_status": item["gold_status"]
    })


g3_summary_df = pd.DataFrame(g3_summary_rows)

g3_summary_df.to_csv(
    G3_SUMMARY_FILE,
    index=False
)


# ------------------------------------------------------------
# 5. Reload and validate saved files
# ------------------------------------------------------------

with open(G3_GOLD_FILE, encoding="utf-8") as f:

    g3_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(g3_reloaded) == 20

g3_ids = [
    item["question_id"]
    for item in g3_reloaded
]

assert len(g3_ids) == len(set(g3_ids))

assert sum(
    item["answerable"]
    for item in g3_reloaded
) == 17

assert sum(
    not item["answerable"]
    for item in g3_reloaded
) == 3


# ------------------------------------------------------------
# 6. Validate abstention questions
# ------------------------------------------------------------

for item in g3_reloaded:

    if not item["answerable"]:

        assert item["abstention_reason"]
        assert item["gold_status"] == "ABSTENTION_GOLD"


# ------------------------------------------------------------
# 7. Validate resolved CFR questions
# ------------------------------------------------------------

cfr_questions = [
    item for item in g3_reloaded
    if item["expected_sections"]
]

assert len(cfr_questions) == 16

for item in cfr_questions:

    assert item["gold_chunk_ids"], (
        f"Missing gold chunks for {item['question_id']}"
    )

    assert item["gold_status"] == "SECTION_RESOLVED"

    assert all(
        chunk_id in {
            str(c.get("chunk_id"))
            for c in g3_chunks
        }
        for chunk_id in item["gold_chunk_ids"]
    )


# ------------------------------------------------------------
# 8. SHA256 helper
# ------------------------------------------------------------

def g3_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 70)
print("COMPLAINTIQ — G3 STRUCTURAL ACCEPTANCE")
print("=" * 70)

print("Total questions:", len(g3_reloaded))
print("Answerable questions:", sum(
    x["answerable"] for x in g3_reloaded
))
print("Abstention questions:", sum(
    not x["answerable"] for x in g3_reloaded
))
print("CFR questions with resolved chunks:", len(cfr_questions))

print("\nQuestion status:")
display(
    g3_summary_df[
        [
            "question_id",
            "regulation",
            "gold_chunk_count",
            "gold_status"
        ]
    ]
)

print("\nSaved artifacts:")

for path in [
    G3_GOLD_FILE,
    G3_CANDIDATES_FILE,
    G3_SUMMARY_FILE
]:

    assert path.exists()

    print("\n", path)
    print("Size:", path.stat().st_size, "bytes")
    print("SHA256:", g3_sha256(path))


pending = [
    item["question_id"]
    for item in g3_reloaded
    if item["gold_status"] == "PENDING_MANUAL_REVIEW"
]

missing = [
    item["question_id"]
    for item in g3_reloaded
    if item["gold_status"] == "MISSING_GOLD_SECTION"
]

print("\nPending manual review:", pending)
print("Missing gold sections:", missing)

assert not missing, (
    "Some CFR questions have no matching corpus chunks."
)

if pending:

    print(
        "\nREVIEW REQUIRED — G3-17 must be manually checked "
        "against the Supervisory Highlights evidence."
    )

    print(
        "The G3 dataset is saved, but final gold acceptance "
        "is pending that review."
    )

else:

    print("\nPASS — G3 GOLD DATASET STRUCTURAL ACCEPTANCE.")